# Macro AI Lakehouse - Bronze Layer Ingestion
Ingests raw geopolitical risk (GPR), daily market feeds (Brent crude, EUR/USD FX, 10Y Treasury yields), and BEA quarterly software and R&D capital expenditure series into the standardized bronze directory.

In [1]:
import os
import requests
import pandas as pd
import yfinance as yf

# Standardize base and data directory paths
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
BRONZE_DIR = os.path.join(DATA_DIR, "bronze")
os.makedirs(BRONZE_DIR, exist_ok=True)


In [2]:
# Ingest GPR Index from Matteo Iacoviello Fed repo
gpr_url = "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls"
gpr_file_path = os.path.join(BRONZE_DIR, "gpr_monthly_raw.xlsx")

try:
    response = requests.get(gpr_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    response.raise_for_status()
    with open(gpr_file_path, "wb") as f:
        f.write(response.content)
    print(f"Saved GPR Index: {gpr_file_path}")
except Exception as e:
    print(f"Network request failed ({e}), checking existing local bronze file...")
    if os.path.exists(gpr_file_path):
        print(f"Using existing GPR bronze file: {gpr_file_path}")
    else:
        raise


Saved GPR Index: C:\Users\aabha\macro_ai_lakehouse\data\bronze\gpr_monthly_raw.xlsx


In [3]:
# Ingest daily financial market feeds via yfinance
tickers = {
    "market_brent_daily.csv": "BZ=F",           # Brent Crude Oil
    "market_eur_usd_daily.csv": "EURUSD=X",       # EUR / USD Spot FX
    "market_treasury_10y_daily.csv": "^TNX"       # 10Y US Treasury Yield
}

for filename, ticker_symbol in tickers.items():
    out_path = os.path.join(BRONZE_DIR, filename)
    try:
        data = yf.download(ticker_symbol, start="2015-01-01", progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data.reset_index(inplace=True)
        data.to_csv(out_path, index=False)
        print(f"Saved {ticker_symbol} -> {out_path} ({len(data)} trading days)")
    except Exception as e:
        print(f"Network request failed for {ticker_symbol} ({e}), checking existing bronze file...")
        if os.path.exists(out_path):
            print(f"Using existing bronze file: {out_path}")
        else:
            raise


Saved BZ=F -> C:\Users\aabha\macro_ai_lakehouse\data\bronze\market_brent_daily.csv (2949 trading days)


Saved EURUSD=X -> C:\Users\aabha\macro_ai_lakehouse\data\bronze\market_eur_usd_daily.csv (3052 trading days)


Saved ^TNX -> C:\Users\aabha\macro_ai_lakehouse\data\bronze\market_treasury_10y_daily.csv (2945 trading days)


In [4]:
# Ingest BEA tech investment series from FRED
series_map = {
    "B985RC1Q027SBEA": "software_investment_billions",
    "Y006RC1Q027SBEA": "rd_investment_billions"
}

dfs = []
for series_id, col_name in series_map.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df_series = pd.read_csv(url)
    df_series.columns = ["date", col_name]
    df_series[col_name] = pd.to_numeric(df_series[col_name], errors="coerce")
    dfs.append(df_series)

df_fred = dfs[0].merge(dfs[1], on="date", how="outer").sort_values("date")
bronze_fred_path = os.path.join(BRONZE_DIR, "fred_tech_investment.csv")
df_fred.to_csv(bronze_fred_path, index=False)
print(f"Saved FRED tech investment series: {bronze_fred_path} ({len(df_fred)} rows)")


Saved FRED tech investment series: C:\Users\aabha\macro_ai_lakehouse\data\bronze\fred_tech_investment.csv (318 rows)
